# 📝 추출 품질 과제 LV2: 평가 결과를 개선 계획으로 연결하기

교안 01의 **기준선 측정**과 교안 02의 **오류 진단·개선 조건·결과 판단**을 연결합니다.  
기존 자가면역 논문 데이터에서 **주석된 두 문서만** 평가하고, 잘못 뽑힌 것과 빠뜨린 것의 근거를 읽습니다.

- 코딩 5문항과 서술형 3문항입니다. 위에서부터 작성합니다.  
- 입력은 `lv2_corpus.jsonl`, `lv2_triples.jsonl`, `lv2_gold.jsonl`, `lv2_gold_checklist.jsonl`입니다.  
- 이번 과제에는 두 번째 실제 추출본이 없습니다. **기준선·오류 근거표·재실행 계획**을 제출하며 모델 호출은 하지 않습니다.  
- 다른 문서의 골드나 교안의 추출 결과로 대체하지 않습니다.

In [ ]:
# [제공 코드]

# 실습에 공통으로 쓸 파일 경로와 읽기, 저장 함수를 준비합니다.

import json
import random
from collections import Counter
from pathlib import Path

data_dir = Path("data")  # 제공된 원문, 추출된 트리플, 골드 파일이 있는 폴더입니다.
output_dir = Path("output")  # 직접 계산한 지표와 검토 기록을 저장할 폴더입니다.
output_dir.mkdir(exist_ok=True)

def load_rows(filename):
    """data 폴더의 JSONL 파일을 딕셔너리 목록으로 읽습니다."""
    return [json.loads(line) for line in (data_dir / filename).read_text(encoding="utf-8").splitlines() if line.strip()]

def write_json(filename, value):
    """이번 실습의 결과를 output 폴더에 JSON으로 저장합니다."""
    (output_dir / filename).write_text(json.dumps(value, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

def write_rows(filename, rows):
    """검토 기록을 한 줄에 한 항목인 JSONL로 저장합니다."""
    content = "\n".join(json.dumps(row, ensure_ascii=False) for row in rows)
    (output_dir / filename).write_text(content + "\n", encoding="utf-8")

def triple_key(row):
    """고유 관계를 비교할 (주어, 관계, 목적어) 튜플을 돌려줍니다."""

    # 평가 전에 표기를 바꾸지 않습니다. 이름 정규화는 다음 단원에서 배웁니다.
    return (row["subject"], row["relation"], row["object"])

In [ ]:
# [제공 코드]

# 논문 추출을 관계, 타입 규칙에 따라 통과와 기각으로 나눌 함수를 준비합니다.

signatures = {
    "TREATS": ("Compound", "Disease"),
    "PALLIATES": ("Compound", "Disease"),
    "BINDS": ("Compound", "Gene"),
    "UPREGULATES_CG": ("Compound", "Gene"),
    "DOWNREGULATES_CG": ("Compound", "Gene"),
    "ASSOCIATES": ("Disease", "Gene"),
    "PRESENTS": ("Disease", "Symptom"),
    "INCLUDES": ("PharmacologicClass", "Compound"),
}

def check_signature(row):
    """관계, 타입 위반 사유를 돌려주고, 통과하면 None을 돌려줍니다."""
    if row["relation"] not in signatures:
        return "허용 관계가 아님"
    # signatures의 값은 해당 관계가 요구하는 주어 타입과 목적어 타입입니다.
    subject_type, object_type = signatures[row["relation"]]
    if row["subject_type"] != subject_type:
        return f"주어 타입이 {subject_type} 이어야 함"
    if row["object_type"] != object_type:
        return f"목적어 타입이 {object_type} 이어야 함"

    return None

def split_schema(rows):
    """추출 목록을 스키마 통과 목록과 사유가 붙은 기각 목록으로 나눕니다."""
    valid, rejected = [], []
    for row in rows:
        reason = check_signature(row)
        if reason is None:
            valid.append(row)
        else:
            # 원본은 유지하고 기각 목록에만 사유를 덧붙입니다.
            rejected.append(dict(row, reject_reason=reason))

    return valid, rejected

In [ ]:
# [제공 코드]

# 추출과 골드를 비교해 TP, FP, FN, 정밀도, 재현율, F1을 계산할 함수를 정의합니다.

def measure_exact(rows, gold_rows):
    """고유 관계의 완전일치 TP, FP, FN과 정밀도, 재현율, F1을 돌려줍니다."""
    # 집합으로 바꿔 같은 관계를 여러 번 뽑아도 한 번만 셉니다.
    predicted = {triple_key(row) for row in rows}
    expected = {triple_key(row) for row in gold_rows}
    tp = len(predicted & expected)  # 추출 결과와 골드 양쪽에 있는 관계입니다.
    fp = len(predicted - expected)  # 골드에 없는 추출입니다. 표기 차이도 포함합니다.
    fn = len(expected - predicted)  # 골드에는 있지만 추출하지 못한 관계입니다.
    # 분모가 없으면 0점 대신 미산출(None)로 남깁니다.
    precision = tp / len(predicted) if predicted else None
    recall = tp / len(expected) if expected else None

    # 골드가 있는데 아무것도 뽑지 않으면 F1은 0입니다. 골드가 없으면 평가에서 별도 표시합니다.
    f1 = 2 * tp / (2 * tp + fp + fn) if expected else None

    return {"predicted": len(predicted), "gold": len(expected), "tp": tp, "fp": fp, "fn": fn,
            "precision": precision, "recall": recall, "f1": f1}

In [ ]:
# [제공 코드] 골드가 없는 네 문서는 평가 대상에 포함하지 않습니다.
corpus = load_rows("lv2_corpus.jsonl")
docs = {row["doc_id"]: row for row in corpus}
triples = load_rows("lv2_triples.jsonl")
valid, rejected = split_schema(triples)
gold = load_rows("lv2_gold.jsonl")
checklist = load_rows("lv2_gold_checklist.jsonl")

# 양성 골드에서 문서 범위를 역으로 만들지 않고 주석 대상 목록을 별도로 둡니다.
# 그래야 나중에 관계가 0건인 주석 문서도 평가 범위에서 사라지지 않습니다.
annotated_doc_ids = {"PMC13493376", "PMC13495420"}
evaluation_relations = {"TREATS", "PALLIATES", "BINDS", "UPREGULATES_CG",
                        "DOWNREGULATES_CG", "ASSOCIATES", "PRESENTS"}
print(f"추출 {len(triples)}행 · 골드 {len(gold)}항목 · 주석 문서 {len(annotated_doc_ids)}편")
for doc_id in sorted(annotated_doc_ids):
    print(doc_id, docs[doc_id]["title"])

**이번 평가의 기준**

- 평가 대상은 지정한 문서와 관계 범위입니다. 범위 밖 결과는 먼저 분리합니다.  
- 한 항목은 고유한 **(주어, 관계, 목적어)**입니다. 같은 관계가 여러 문서에 있어도 한 번만 셉니다. 출처와 근거는 별도 기록으로 보존합니다.  
- 세 값이 **문자열까지 모두 같을 때** 일치로 셉니다. 대소문자·괄호가 다르면 다른 항목입니다.  
- 이 점수는 **저장된 골드와의 완전일치**입니다. FP라고 해서 반드시 원문의 의미를 틀리게 읽었다는 뜻은 아닙니다. 원문 근거와 표기 차이를 함께 확인합니다.  
- 스키마 검사, 근거 원문 일치 검사, 사람이 판정한 근거 적합 여부는 서로 다른 검사입니다. 각각의 대상과 분모를 적습니다.

**이 골드의 주석 기준을 읽으세요**

관계 7종과 문서 2편의 범위를 유지합니다. 개별 약물·질병·유전자·증상만 인정하며, 약효군과 여러 유전자를 아우르는 계열 이름은 제외합니다. BINDS에는 문서가 설명하는 표적·대사 효소·수송체 관계도 포함합니다.

기존 골드는 문서에 적힌 이름 중 주석자가 선택한 표기를 사용했습니다. 예를 들어 `Methotrexate`와 `MTX`, `prednisone`과 `Prednisone`은 의미가 가까워도 **이번 완전일치 계산에서는 다른 문자열**입니다. 평가 도중 이름을 바꾸어 점수를 높이지 말고, 차이를 오류 근거표에 기록합니다. 표기를 통일하는 방법은 다음 단원에서 다룹니다.

골드는 절대적인 의학 지식이 아니라 이 문서·주석 지침·표기 선택에 따른 정답표입니다. 근거를 읽어 이견을 남길 수 있지만, 이번 기준선을 계산하는 중에는 골드를 변경하지 않습니다.

## 1. 주석 범위에 맞춰 평가 입력을 준비하세요

**배경**: 전체 추출은 여섯 문서에서 나왔지만 골드는 두 문서만 주석했습니다. 범위를 일치시킨 뒤 고유 관계로 평가합니다.

**요구사항**

- **`evaluation_plan`**을 딕셔너리로 만드세요. **`doc_ids`**는 annotated_doc_ids를 정렬한 리스트, **`relations`**는 evaluation_relations를 정렬한 리스트입니다.  
- **`unit`**은 `"unique_triple"`, **`matching`**은 `"exact"`, **`gold_file`**은 `"lv2_gold.jsonl"`입니다.  
- **`scoped`**는 valid에서 해당 문서와 관계에 속하는 원본 행만 남긴 리스트입니다. 순서와 중복 근거를 유지하세요.  
- **`scope_gold`**는 gold에서 관계가 평가 범위에 있고 sources의 모든 doc_id가 주석 문서 목록에 드는 항목 리스트입니다.

**확인 기준**: scoped는 9행이고 scope_gold는 7항목입니다. 같은 관계가 근거만 다르게 두 번 나온 행은 이 단계에서 보존합니다.

<details><summary>힌트</summary>

**접근방법**: 평가 계획을 적고 추출과 골드에 동일한 문서·관계 범위를 적용합니다.

**세부구현**

1. 정렬한 범위를 계획에 저장하세요.  
2. 원본 추출의 source_doc_id와 relation을 함께 확인하세요.  
3. 골드 sources는 첫 항목만 보지 않고 모두 확인하세요.

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert evaluation_plan == {"doc_ids": sorted(annotated_doc_ids), "relations": sorted(evaluation_relations),
                           "unit": "unique_triple", "matching": "exact", "gold_file": "lv2_gold.jsonl"}
assert scoped == [row for row in valid if row["source_doc_id"] in annotated_doc_ids and row["relation"] in evaluation_relations]
assert scope_gold == [row for row in gold if row["relation"] in evaluation_relations
                      and all(source["doc_id"] in annotated_doc_ids for source in row["sources"])]
assert len(scoped) == 9 and len(scope_gold) == 7
print("✅ 평가 범위 통과!")

## 2. 주석한 문장 목록을 검산하세요

**배경**: 검토 기록 30개가 있다는 것만으로 30문장을 모두 봤다고 할 수는 없습니다. 실제 문서·문장 번호의 집합을 대조합니다.

**요구사항**

- **`expected_sentences`**를 주석된 두 문서의 `(doc_id, sent_id)` 튜플 집합으로 만드세요. sent_id는 0부터 시작합니다.  
- **`reviewed_sentences`**는 checklist의 같은 튜플 집합입니다.  
- **`missing_sentences`**에는 예상 목록에서 검토 목록을 뺀 차집합, **`extra_sentences`**에는 반대 차집합을 담으세요.  
- **`missing_notes`**는 checklist 중 n_gold가 0이고 note가 공백뿐인 행의 리스트입니다.  
- 문장 검토 완료가 관계 누락 0건을 보장하는지는 출력문 한 줄로 설명하세요.

**확인 기준**: 두 집합은 같은 30개이며 차집합은 모두 비어 있습니다. missing_notes도 빈 리스트입니다. 검토 완료와 내용 완전성은 다른 주장입니다.

<details><summary>힌트</summary>

**접근방법**: 개수만 비교하지 않고 실제 문장 식별자의 차집합을 검사합니다.

**세부구현**

1. 각 주석 문서의 sentences 길이로 문장 번호를 만드세요.  
2. 두 방향 차집합을 계산하세요.  
3. 0건 문장의 사유에는 strip을 적용하세요.

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert expected_sentences == {(doc_id, i) for doc_id in annotated_doc_ids for i in range(len(docs[doc_id]["sentences"]))}
assert reviewed_sentences == {(row["doc_id"], row["sent_id"]) for row in checklist}
assert missing_sentences == expected_sentences - reviewed_sentences == set()
assert extra_sentences == reviewed_sentences - expected_sentences == set()
assert len(expected_sentences) == 30
assert missing_notes == [row for row in checklist if row["n_gold"] == 0 and not row["note"].strip()] == []
print("✅ 주석 기록 검산 통과!")

## 3. 같은 TP로 P/R/F1을 계산하세요

**배경**: 이전 과제의 사람 점수를 정밀도 자리에 섞지 않습니다. 동일한 고유 관계 교집합에서 세 비율을 계산합니다.

**요구사항**

- **`predicted_keys`**, **`gold_keys`**를 scoped와 scope_gold의 triple_key 집합으로 만드세요.  
- **`tp_keys`**, **`fp_keys`**, **`fn_keys`**에는 교집합·추출에만 있는 차집합·골드에만 있는 차집합을 각각 담으세요.  
- **`baseline`**에는 measure_exact로 계산한 딕셔너리를 담으세요.  
- **`duplicate_rows`**에는 scoped 행 수에서 고유 추출 관계 수를 뺀 정수를 담으세요.  
- TP·FP·FN·P/R/F1과 중복 행 수를 출력하세요.

**확인 기준**: 고유 추출 8건, 골드 7건, TP=2·FP=6·FN=5입니다. P=0.25, R=2/7, F1=4/15이며 중복 행은 1개입니다.

<details><summary>힌트</summary>

**접근방법**: 집합으로 오류 목록을 만들고 같은 기준의 지표를 제공 함수로 계산합니다.

**세부구현**

1. 타입과 근거는 triple_key에 포함하지 않습니다.  
2. 차집합의 방향을 구분하세요.  
3. 행 수와 고유 관계 수를 따로 출력하세요.

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert predicted_keys == {triple_key(row) for row in scoped} and gold_keys == {triple_key(row) for row in scope_gold}
assert tp_keys == predicted_keys & gold_keys
assert fp_keys == predicted_keys - gold_keys and fn_keys == gold_keys - predicted_keys
assert baseline == measure_exact(scoped, scope_gold)
assert (baseline["predicted"], baseline["gold"], baseline["tp"], baseline["fp"], baseline["fn"]) == (8, 7, 2, 6, 5)
assert abs(baseline["precision"] - 0.25) < 1e-12
assert abs(baseline["recall"] - 2 / 7) < 1e-12 and abs(baseline["f1"] - 4 / 15) < 1e-12
assert duplicate_rows == len(scoped) - len(predicted_keys) == 1
assert measure_exact(scoped + scoped, scope_gold) == baseline, "근거 중복으로 지표가 바뀌면 안 됩니다."
print("✅ 완전일치 기준선 통과!")

## 4. FP와 FN의 원문 근거표를 만드세요

**배경**: 점수만으로 수정 방향을 정할 수 없습니다. FP에는 모델이 적은 근거를, FN에는 골드 주석의 근거를 붙여 읽습니다.

**요구사항**

- **`fp_evidence`**를 scoped에서 fp_keys에 속하는 각 행을 변환한 딕셔너리 리스트로 만드세요. 원래 순서와 중복 근거를 보존합니다.  
- 각 항목의 **`key`**는 triple_key를 리스트로 바꾼 값, **`doc_id`**는 source_doc_id, **`evidence`**는 모델의 근거 문자열입니다.  
- 불리언 **`verbatim`**은 그 근거 전체가 해당 원문에 있는지, 문자열 **`status`**는 `"검토 필요"`입니다.  
- **`fn_evidence`**는 scope_gold 중 fn_keys에 드는 각 항목의 딕셔너리 리스트입니다. **`key`**는 키 리스트, **`sources`**는 골드의 sources 전체, **`status`**는 `"검토 필요"`입니다.  
- 두 목록을 출력하세요. 원문 불일치만으로 FP의 원인을 확정하지 마세요.

**확인 기준**: FP는 고유 6건이지만 근거표는 7행입니다. FN 근거표는 5행이고 각 항목의 출처를 보존합니다.

<details><summary>힌트</summary>

**접근방법**: 오류 집합으로 원본 행을 고르고 각 행의 출처 정보를 유지합니다.

**세부구현**

1. FP 행을 중복 제거하지 마세요.  
2. 출처 문서 한 편에서만 근거를 대조하세요.  
3. FN은 sources의 첫 원소가 아니라 전체를 남기세요.

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
expected_fp = [{"key": list(triple_key(row)), "doc_id": row["source_doc_id"], "evidence": row["evidence"],
                "verbatim": row["evidence"] in docs[row["source_doc_id"]]["text"], "status": "검토 필요"}
               for row in scoped if triple_key(row) in fp_keys]
assert fp_evidence == expected_fp, "원본 순서와 모든 근거를 보존하세요."
assert fn_evidence == [{"key": list(triple_key(row)), "sources": row["sources"], "status": "검토 필요"}
                       for row in scope_gold if triple_key(row) in fn_keys]
assert len(fp_evidence) == 7 and len(fn_evidence) == 5
assert {tuple(row["key"]) for row in fp_evidence} == fp_keys
print("✅ 오류 근거표 통과!")

## 5. 오류 세 건을 서로 다른 원인으로 진단하세요

**배경**: FP·FN이라는 평가 분류와 실제 오류 원인은 다릅니다. 표기 차이와 내용 누락을 구분합니다.

**요구사항**

- **오류 진단표**를 작성하세요. 열은 `대상 / 원문 근거 / 완전일치에서의 상태 / 원인 / 다음 확인`입니다.  
- `(Prednisone, TREATS, SLE)`와 골드의 `(prednisone, TREATS, SLE)`를 한 행에서 비교하세요. 문서 PMC13495420의 문장 12를 읽습니다.  
- `(Corticosteroids, TREATS, SLE)`를 두 번째 행에 적고 같은 문장과 약효군 제외 기준을 연결하세요.  
- 골드의 `(hydroxychloroquine, TREATS, SLE)`를 세 번째 행에 적고 문장 10 및 골드 judgment를 읽으세요. 문맥에서 해석한 골드 관계라는 점과 추출 목록에 없는 점을 함께 적습니다.  
- 첫 번째 사례를 고칠 때 프롬프트 수정과 다음 단원의 이름 정규화를 구분하세요.

**확인 기준**: 표기 차이는 FP와 FN을 동시에 만들 수 있고, 약효군은 개체 범위 문제이며, 골드의 문맥 추론은 지침 검토도 필요하다고 설명합니다. 자동 채점은 하지 않습니다. 작성한 뒤 정답 노트북의 모범 서술과 비교하세요.

<details><summary>힌트</summary>

**접근방법**: 원문 구절, 판정 기준, 결론을 순서대로 연결하세요.

**세부구현**

1. 판단에 사용한 원문 표현을 짧게 인용하세요.  
2. 그 표현이 어떤 판정 기준을 충족하거나 충족하지 못하는지 설명하세요.  
3. 점수의 문제와 추출 내용의 문제를 구분하세요.

</details>

*(여기에 원문 근거와 판단을 작성하세요.)*

## 6. 골드 변경과 추출기 개선을 구분하세요

**배경**: 평가 도중 골드를 바꾸면 개선 전후가 같은 시험을 치른 것이 아니게 됩니다.

**요구사항**

- **판정 메모**에 다음 상황을 설명하세요. 팀원이 Corticosteroids도 개체로 인정하자고 제안했습니다.  
- 원래 지침을 유지할 때 해당 항목의 상태를 적으세요.  
- 제안을 받아들일 때 바꿔야 하는 주석 지침·골드·평가 버전을 적으세요. 관계 수만 더하고 끝내면 안 되는 이유도 설명하세요.  
- 기존 점수와 새 기준 점수를 같은 개선 전후 표에 직접 넣어도 되는지 판단하세요.  
- 점수가 어느 방향으로 얼마나 변하는지는 실제 전체 재계산 전까지 단정하지 마세요.

**확인 기준**: 지침을 바꾸는 의사결정과 프롬프트를 고치는 실험을 구분하며, 기준 변경 시 양쪽 결과를 새 기준으로 다시 평가합니다. 자동 채점은 하지 않습니다. 작성한 뒤 정답 노트북의 모범 서술과 비교하세요.

<details><summary>힌트</summary>

**접근방법**: 원문 구절, 판정 기준, 결론을 순서대로 연결하세요.

**세부구현**

1. 판단에 사용한 원문 표현을 짧게 인용하세요.  
2. 그 표현이 어떤 판정 기준을 충족하거나 충족하지 못하는지 설명하세요.  
3. 점수의 문제와 추출 내용의 문제를 구분하세요.

</details>

*(여기에 원문 근거와 판단을 작성하세요.)*

## 7. 한 가지 수정의 재실행 계획을 작성하세요

**배경**: 현재 폴더에는 이 과제의 개선 추출본이 없습니다. 실제로 수행할 수 있는 비교 계획을 만들고 완료된 실험과 구분합니다.

**요구사항**

- **실험 계획**에 `가설 / 바꿀 것 한 가지 / 고정 조건 / 새 결과 평가 방법 / 채택 기준 / 현재 상태`를 적으세요.  
- 원인 진단에 근거해 문맥 제공 또는 개체 범위 지시 중 한 가지만 선택합니다. 문맥을 추가한다면 추출 대상 문장과 참고 문맥을 구분하세요.  
- 문서·관계 범위·골드·일치 규칙·모델 설정을 고정합니다. 이름 정규화로 점수를 바꾸는 작업은 이번 실험에 섞지 않습니다.  
- 개선판의 **새 추출 전체**를 같은 범위로 걸러 measure_exact로 평가한다고 적으세요. 옛 표본에 남은 키만 고르지 않습니다.  
- FP/FN과 P/R/F1을 함께 보고 대표 오류의 원문을 다시 확인하는 채택 기준을 세우세요.  
- 현재 상태는 `미실행`입니다. 이번 과제에서 API 호출을 수행할 필요는 없습니다.

**확인 기준**: 바뀌는 조건은 하나이고, 같은 평가 기준으로 새 추출 전체를 비교합니다. 효과를 이미 얻었다고 서술하지 않습니다. 자동 채점은 하지 않습니다. 작성한 뒤 정답 노트북의 모범 서술과 비교하세요.

<details><summary>힌트</summary>

**접근방법**: 원문 구절, 판정 기준, 결론을 순서대로 연결하세요.

**세부구현**

1. 판단에 사용한 원문 표현을 짧게 인용하세요.  
2. 그 표현이 어떤 판정 기준을 충족하거나 충족하지 못하는지 설명하세요.  
3. 점수의 문제와 추출 내용의 문제를 구분하세요.

</details>

*(여기에 원문 근거와 판단을 작성하세요.)*

## 8. 기준선과 미실행 상태를 보고서로 저장하세요

**배경**: 검토자가 무엇을 실제로 측정했고 무엇이 계획인지 알 수 있어야 합니다.

**요구사항**

- **`quality_report`**를 딕셔너리로 만드세요. **`evaluation`**에는 evaluation_plan, **`baseline`**에는 baseline을 넣습니다.  
- **`review`**는 `{"fp_unique": FP 고유 관계 수, "fp_evidence_rows": FP 근거 행 수, "fn_unique": FN 고유 관계 수, "duplicate_rows": 중복 행 수}` 딕셔너리입니다.  
- **`improved`**는 `None`, **`experiment_status`**는 `"미실행"`, **`decision`**은 `"판단 보류"`로 적으세요.  
- quality_report를 **`lv2_quality_report.json`**, fp_evidence를 **`lv2_fp_evidence.json`**, fn_evidence를 **`lv2_fn_evidence.json`**으로 저장하세요.  
- 서술형 답안 세 개는 이 노트북에 함께 보관합니다.

**확인 기준**: 저장한 지표는 P=.25, R=2/7, F1=4/15입니다. 개선판 점수는 비어 있고 실제 재실행이 필요하다고 표시됩니다.

<details><summary>힌트</summary>

**접근방법**: 계산한 기준선과 검토할 근거를 저장하되 실행하지 않은 결과는 비워 둡니다.

**세부구현**

1. 지표의 원래 값을 저장하세요.  
2. 고유 관계 수와 근거 행 수를 구분하세요.  
3. write_json으로 저장한 파일을 다시 읽어 확인하세요.

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert quality_report["evaluation"] == evaluation_plan and quality_report["baseline"] == baseline
assert quality_report["review"] == {"fp_unique": len(fp_keys), "fp_evidence_rows": len(fp_evidence),
                                   "fn_unique": len(fn_keys), "duplicate_rows": duplicate_rows}
assert quality_report["improved"] is None
assert quality_report["experiment_status"] == "미실행" and quality_report["decision"] == "판단 보류"
assert json.loads((output_dir / "lv2_quality_report.json").read_text()) == quality_report
assert json.loads((output_dir / "lv2_fp_evidence.json").read_text()) == fp_evidence
assert json.loads((output_dir / "lv2_fn_evidence.json").read_text()) == fn_evidence
print("✅ LV2 기준선·오류 근거표 저장 통과!")